In [ ]:
%load_ext autoreload
%autoreload 2

# Deterministic Sankey from method

In [ ]:
# Generate sbm and adjacency matrix
import sys
sys.path.append('..')
from pygraphon.graphons import Graphon
from pygraphon.graphons import StepGraphon
from experiments.utils_generators import AssociativeFullyRandom
k = 2
n = 400
# theta init
theta_intra_min, theta_intra_max = 0.4, 0.8
theta_inter_min, theta_inter_max = 0.1, 0.3
import numpy as np
np.random.seed(420)
theta = AssociativeFullyRandom(
                    K=k,
                    min_inside_classes=theta_intra_min,
                    max_inside_classes=theta_intra_max,
                    min_outside_classes=theta_inter_min,
                    max_outside_classes=theta_inter_max,
                )
graphon = StepGraphon(theta.Theta, bandwidthHist=1 / k)
adjacency_matrix=graphon.draw(n=n, exchangeable=False)
print(graphon.get_theta())

In [ ]:
#import smoothnethist
from pygraphon.estimators import SmoothNetHist

est = SmoothNetHist(bandwidth=1/(2*k))
est.fit(adjacency_matrix)

In [ ]:
import numpy as np
list_graphons = [est.all_cluster_graphons[i].get_theta()[np.triu_indices(est.all_cluster_graphons[i].get_theta().shape[0])] for i in range(len(est.all_cluster_graphons))]

sort_indices = np.argsort(list_graphons[-1])

for i in range(len(list_graphons)):
    list_graphons[i] = list_graphons[i][sort_indices]
graphons_array = np.array(list_graphons)
graphons_array = np.rot90(np.insert(graphons_array, 0, np.ones(graphons_array.shape[1]), axis=0), k=3)

In [ ]:
#makes list into pandas dataframe
import pandas as pd
pd.set_option('display.float_format', '{:.3f}'.format)
columns = []
for i in range(len(list_graphons)):
    columns.append("SmoothNetHist_" + str(len(list_graphons)-i))
columns.append("Count")

def list_to_df(array, columns):
    df = pd.DataFrame(array, columns=columns)
    return df

df = list_to_df(graphons_array, columns=columns)
df

# Plotting sankey

In [ ]:
import sys
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
def find_peak_index(lst):
    for i in range(1, len(lst)):
        if lst[i] < lst[i - 1]:
            return i - 1
    return len(lst) - 1

In [ ]:
source_nodes = [0.129, 0.177, 0.188,0.201,0.627,0.642,0.70001,0.722,0.749,0.785, 0.183, 0.634, 0.189, 0.711, 0.767, 0.174, 0.739, 0.704, 0.435]
idx = find_peak_index(source_nodes)
print("The original grapghon has", idx+1, "nodes")
# the 0.0001 is to avoid the error of having nodes one the boundaries of [0,1], somehow it messes up the visualization
source_intervals = [0.0001 if i < idx+1 else (i-idx)/len(source_nodes[idx+1:]) for i in range(len(source_nodes))]
nodes = [{"label": str(label)+'_'+str(i)} for i, label in enumerate(sorted(source_nodes))]
intervals_x = [interval for _, interval in sorted(zip(source_nodes, source_intervals))]
intervals_x[intervals_x.index(1)] -= 0.0001
intervals_y = [(i)/len(nodes) for i in range(len(nodes))]
intervals_y[intervals_y.index(0)] += 0.0001
diff_intervals_y = [intervals_y[i+1] - intervals_y[i] for i in range(len(intervals_y)-1)]

#Need to correct the y interval corresponding to 0.70001 because the sankey than is misplaced vertically
y_70001 = intervals_y[sorted(source_nodes).index(0.70001)]
y_704 = intervals_y[sorted(source_nodes).index(0.704)]
intervals_y[sorted(source_nodes).index(0.70001)] = y_704
intervals_y[sorted(source_nodes).index(0.704)] = y_70001

In [ ]:
links = [
    {'source': 2, 'target': 3, 'value': 1},
    {'source': 4, 'target': 3, 'value': 1},
    {'source': 8, 'target': 9, 'value': 1},
    {'source': 10, 'target': 9, 'value': 1},
    {'source': 3, 'target': 5, 'value': 2},
    {'source': 6, 'target': 5, 'value': 1},
    {'source': 11, 'target': 13, 'value': 1},
    {'source': 14, 'target': 13, 'value': 1},
    {'source': 16, 'target': 17, 'value': 1},
    {'source': 18, 'target': 17, 'value': 1},
    {'source': 0, 'target': 1, 'value': 1},
    {'source': 5, 'target': 1, 'value': 3},
    {'source': 13, 'target': 15, 'value': 2},
    {'source': 17, 'target': 15, 'value': 2},
    {'source': 9, 'target': 12, 'value': 2},
    {'source': 15, 'target': 12, 'value': 4},
    {'source': 12, 'target': 7, 'value': 6},
    {'source': 1, 'target': 7, 'value': 4},
]

In [ ]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import plotly.io as pio
# Define the data

#sorted_links = sorted(links, key=lambda k: k['source']) #this doesnt chnage anything
# Create the figure and subplot
fig = make_subplots(rows=1, cols=1)

# Add the Sankey trace
fig.add_trace(go.Sankey(
    arrangement = "perpendicular",
    node=dict(
        pad=10*len(nodes),  # Adjust the padding between nodes
        thickness=20,  # Adjust the thickness of nodes
        line=dict(color='black', width=1),  # Define the color and width of node lines
        label=[f"{float(node['label']):.3f}" for node in nodes],  # Extract node labels from the 'nodes' list
        x = intervals_x,
        y = intervals_y,
    ),
    link=dict(
        source=[link['source'] for link in links],  # Extract source indices from the 'links' list
        target=[link['target'] for link in links],  # Extract target indices from the 'links' list
        value=[link['value'] for link in links],  # Extract values from the 'links' list
    )
))

# Update the layout
fig.update_layout(title_text='Sankey Diagram')
# Save the Sankey diagram as SVG
pio.write_image(fig, 'sankey.svg', format='svg')

# Save the Sankey diagram as PNG
pio.write_image(fig, 'sankey.png', format='png')

# Save the Sankey diagram as PDF
pio.write_image(fig, 'sankey.pdf', format='pdf')

# Show the plot
fig.show()



# Getting SSM from each step of the Sankey

In [ ]:
#Plotting each graphon in subplots
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, len(est.all_cluster_graphons), figsize=(20, 5))
nb_smoothnet_hist = len(est.all_cluster_graphons)
#cbar_ax = fig.add_axes([-0.05, 0.15, 0.05, 0.7])
for i in range(nb_smoothnet_hist):
    axs[i].imshow(est.all_cluster_graphons[nb_smoothnet_hist-i-1].get_theta(), cmap='jet', interpolation='nearest', vmin=0, vmax=1)
    axs[i].set_title("SSM_" + str(nb_smoothnet_hist-i))
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    plt.tight_layout()
#plt.colorbar(axs[i].imshow(est.all_cluster_graphons[nb_smoothnet_hist-i-1].get_theta(), cmap='jet', interpolation='nearest', vmin=0, vmax=1), cax=cbar_ax)

In [ ]:
#Get SSM from selected one
selected_graphon = est.best_pairs['bic']["graphon"].get_theta()

fig, axs = plt.subplots(1, 1, figsize=(20, 5))
axs.imshow(selected_graphon, cmap='jet', interpolation='nearest', vmin=0, vmax=1)
axs.set_title("OptSSM")
axs.set_xticks([])
axs.set_yticks([])
plt.tight_layout()